# Sesión 2 — Demo: RAG sobre Reviews de Baldosas Cerámicas (Ceramic Tiles)

Notebook de demostración para la **Sesión 2** del curso de IA Generativa (ISDI MDA).

**Contenido:**
1. Configuración del modelo (Gemini o Ollama — una sola celda)
2. Carga de datos: 300 reviews de Amazon Home and Kitchen (Ceramic Tiles)
3. Construcción del índice vectorial en ChromaDB
4. Baseline: preguntar al LLM sin contexto
5. RAG: preguntar con contexto recuperado
6. Comparación lado a lado

---

> **Backend `gemini`**: ejecutar en Google Colab con una API key en Secrets.  
> **Backend `ollama`**: ejecutar en local con Ollama corriendo (`ollama serve`).  
> El resto del notebook no cambia.

In [ ]:
%pip install -q \
    langchain \
    langchain-google-genai \
    langchain-ollama \
    langchain-chroma \
    chromadb \
    "datasets<3.0" \
    pandas

## 1. Configuración del modelo

Cambiad `BACKEND` para alternar entre Gemini y Ollama. **Solo esta celda diferencia los dos entornos.**

In [ ]:
# ── Configuración ────────────────────────────────────────────────────────────
BACKEND = "ollama"          # "gemini" | "ollama"
OLLAMA_MODEL = "qwen3.5:2b"
OLLAMA_EMBEDDING_MODEL = "nomic-embed-text-v2-moe"
# ─────────────────────────────────────────────────────────────────────────────

if BACKEND == "gemini":
    GOOGLE_API_KEY = "your-api-key-here"

    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=GOOGLE_API_KEY
    )
    print("✓ Backend: Gemini 2.5 Flash")

elif BACKEND == "ollama":
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL)
    
    print(f"✓ Backend: Ollama — {OLLAMA_MODEL}")

else:
    raise ValueError(f"Backend desconocido: {BACKEND}. Usa 'gemini' o 'ollama'.")

from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model=OLLAMA_EMBEDDING_MODEL)
print(f"✓ Embeddings: Ollama — {OLLAMA_EMBEDDING_MODEL}")

## 2. Carga de datos

Usamos las primeras 300 reviews del dataset de Amazon Home and Kitchen, enfocándonos en productos relacionados con baldosas cerámicas y mejoras del hogar.

In [ ]:
from datasets import load_dataset
import pandas as pd

print("Cargando dataset de Home and Kitchen...")
ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Home_and_Kitchen",
    split="full",
    streaming=True,
    trust_remote_code=True,
)

# Filtrar reviews relacionadas con baldosas cerámicas, azulejos, tools, etc.
keywords = ["tile", "tiles", "ceramic", "grout", "bathroom", "kitchen", "floor", "wall"]

reviews = []
for item in ds:
    if len(reviews) >= 300:
        break
    
    text = str(item.get("text", "")).lower()
    title = str(item.get("title", "")).lower()
    
    # Incluir si contiene alguna palabra clave
    if any(keyword in text or keyword in title for keyword in keywords):
        reviews.append(item)

df = pd.DataFrame(reviews)[["title", "text", "rating", "parent_asin"]].dropna(subset=["text"])
df["rating"] = df["rating"].astype(int)

print(f"✓ {len(df)} reviews cargadas relacionadas con baldosas cerámicas y mejoras del hogar")
print(f"Distribución de ratings:\n{df['rating'].value_counts().sort_index()}")
print(f"\nEjemplo:")
print(df.iloc[0][["rating", "title", "text"]])

## 3. Construcción del índice vectorial

Cada review se convierte en un `Document` de LangChain y se indexa en ChromaDB.  
La metadata (`rating`, `parent_asin`) permite filtrar después.

In [ ]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

# Convertir filas del dataframe a Documents
docs = [
    Document(
        page_content=f"{row['title']}\n\n{row['text']}",
        metadata={
            "rating": row["rating"],
            "parent_asin": row["parent_asin"]
        }
    )
    for _, row in df.iterrows()
]

print(f"Indexando {len(docs)} documentos en ChromaDB...")
print("(Puede tardar 1-2 minutos mientras se generan los embeddings)")

vectorstore = Chroma.from_documents(docs, embeddings)

print(f"✓ Índice creado con {vectorstore._collection.count()} vectores")

## 4. Baseline: preguntar al LLM sin contexto

Preguntamos directamente al modelo, sin ninguna información del catálogo.  
El modelo responde solo con lo que aprendió durante el entrenamiento.

In [ ]:
PREGUNTA = "¿Qué problemas comunes tienen las baldosas cerámicas según los clientes?"

respuesta_sin_rag = llm.invoke(PREGUNTA)

print("=== SIN RAG ===")
print(respuesta_sin_rag.content)

## 5. RAG: preguntar con contexto recuperado

La cadena RAG recupera las reviews más relevantes y las inyecta en el prompt antes de generar la respuesta.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

prompt_rag = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente experto en análisis de productos para el hogar, especialmente baldosas cerámicas y mejoras del hogar.
Responde usando SOLO la información de las siguientes reviews de clientes reales.
Si la información no aparece en las reviews, dilo explícitamente.
Cita ejemplos concretos de las reviews cuando sea relevante.

Reviews recuperadas:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('rating', '?')}★] {d.page_content[:400]}"
        for d in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

print("✓ Cadena RAG construida")

In [ ]:
respuesta_con_rag = rag_chain.invoke(PREGUNTA)

print("=== CON RAG ===")
print(respuesta_con_rag)

## 6. Comparación y exploración

Veamos qué documentos recuperó el retriever, y probemos otras preguntas.

In [ ]:
# Inspeccionar los documentos recuperados
docs_recuperados = retriever.invoke(PREGUNTA)

print(f"Documentos recuperados para: '{PREGUNTA}'\n")
for i, doc in enumerate(docs_recuperados, 1):
    print(f"--- Documento {i} [{doc.metadata.get('rating', '?')}★] ---")
    print(doc.page_content[:250])
    print()

In [ ]:
# Otras preguntas para explorar
otras_preguntas = [
    "¿Qué baldosas tienen mejor valoración en durabilidad?",
    "¿Los clientes mencionan problemas con la instalación?",
    "¿Qué opinan sobre la calidad del material cerámico?",
    "¿Hay quejas sobre el grosor o resistencia de las baldosas?",
    "¿Qué productos recomiendan para cocinas vs baños?"
]

for pregunta in otras_preguntas:
    print(f"\nP: {pregunta}")
    print(f"R: {rag_chain.invoke(pregunta)[:300]}...")
    print("-" * 60)

In [ ]:
# Comparación final: misma pregunta, dos enfoques
pregunta_final = "¿Vale la pena comprar baldosas cerámicas autoadhesivas según los clientes?"

print("PREGUNTA:", pregunta_final)
print()
print("SIN RAG:")
print(llm.invoke(pregunta_final).content[:400])
print()
print("CON RAG:")
print(rag_chain.invoke(pregunta_final)[:400])

---

## Resultados y Análisis

Este notebook demuestra cómo RAG mejora las respuestas del LLM al trabajar con reviews de productos específicos:

- **Sin RAG**: El modelo da respuestas genéricas basadas en su conocimiento general
- **Con RAG**: El modelo proporciona información específica basada en las opiniones reales de los clientes

### Aplicaciones en el sector de mejoras del hogar:
- Análisis de satisfacción del cliente sobre baldosas y azulejos
- Identificación de problemas comunes de instalación
- Comparación de productos cerámicos
- Recomendaciones basadas en casos de uso (cocina, baño, suelos, paredes)

### Próximos pasos:
1. Expandir el dataset con más reviews específicas de baldosas
2. Agregar filtros por tipo de producto (azulejos de pared, suelo, decorativos)
3. Implementar análisis de sentimiento por características (durabilidad, instalación, diseño)
4. Crear un sistema de recomendación basado en las reviews